# 02 · Tienda simulada (precios, stock, carritos, cotizaciones y pedidos)

**Laboratorio 4 · Asistente de compras por voz** (MCD8016 Deep Learning, UTEC)

El enunciado pide una tienda con *"precios, existencias, carritos y pedidos asociados a los productos del catálogo"*, con datos ficticios generados **con una semilla fija** y **restaurables**. Además exige *"validar cantidades, presupuesto y existencias mediante código"* y, al confirmar, *"comprobar nuevamente las condiciones y actualizar pedido e inventario sin cambios parciales ni pedidos duplicados"*.

Este notebook muestra cómo se cumple cada requisito. El código está en `src/asistente_compras/store.py` y lo cubren las pruebas de `tests/test_store.py`.

| Decisión | Por qué |
|---|---|
| **SQLite** (archivo local en Colab) | La interacción es de un usuario y por turnos. La atomicidad la dan las transacciones, no el motor; Postgres no aportaría nada aquí |
| **Dinero en céntimos enteros** | Evita los errores de coma flotante (0,1 + 0,2 ≠ 0,3) en totales y presupuestos |
| **Validaciones en Python, no en el LLM** | Un modelo de 7B puede equivocarse en aritmética; el código no. El LLM solo **explica** los errores que devuelve la tienda |
| **Precio y stock solo desde la BD** | El índice RAG tiene texto, no precios. Así el agente nunca cita un precio o stock desactualizado |
| **`BEGIN IMMEDIATE` + `UNIQUE(quote_id)`** | Confirmación *todo o nada* y sin pedidos duplicados |
| **Bitácora `ops_log`** | El enunciado pide conservar las operaciones ejecutadas en cada escenario |

## 0. Entorno

In [ ]:
# ── Configuración del entorno (idéntica en todos los notebooks) ──────────────────────────
# Detectar si corre en Colab o en la Mac y dejar importable el paquete `asistente_compras`.
import os
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    # 1) Montar Drive: guarda el código y los datos entre sesiones (Colab borra /content al cerrar).
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/lab4/lab4_solution")
    os.environ["LAB4_DATA_DIR"] = "/content/drive/MyDrive/lab4/data"
    # 2) Instalar el paquete y sus dependencias. EXTRAS cambia según el notebook
    #    ("" = núcleo; "rag", "llm", "voz" en los siguientes).
    EXTRAS = ""
    target = f"{PROJECT_DIR}[{EXTRAS}]" if EXTRAS else str(PROJECT_DIR)
    get_ipython().run_line_magic("pip", f"install -q -e {target}")
else:
    # En la Mac, usar como kernel el entorno uv del proyecto (ver README).
    PROJECT_DIR = Path.cwd().resolve().parent

# Una instalación editable recién hecha no se ve hasta reiniciar el kernel;
# agregar src/ al path para evitar ese reinicio.
sys.path.insert(0, str(PROJECT_DIR / "src"))
print(f"Colab: {EN_COLAB} | proyecto: {PROJECT_DIR}")

In [ ]:
import pandas as pd

from asistente_compras.config import get_paths
from asistente_compras.store import Store, StoreConfig, fmt_eur, restore_store

paths = get_paths().ensure()
catalog = pd.read_parquet(paths.processed / "catalog.parquet")  # generado por el notebook 01
store_cfg = StoreConfig()
store_cfg

## 1. Generación y restauración

`restore_store` borra la base y la regenera desde los `product_id` del catálogo:
- **Precio** log-normal con mediana ≈ 20 €, recortado a [1,99 €; 1 500 €] y terminado en ,99. La forma log-normal imita un catálogo real: muchos productos baratos y pocos caros.
- **Stock:** 10 % agotado, 20 % con 1–3 unidades (para probar los límites de cantidad) y el resto con 4–40.

Devuelve una **huella** del inventario. Misma semilla ⇒ misma huella, en la Mac y en Colab.

> **Limitación declarada:** los precios no dependen de la categoría del producto (un libro y un televisor siguen la misma distribución). No afecta a la evaluación, que mide la lógica del agente y no el realismo de los precios.

In [ ]:
huella_inicial = restore_store(paths.store_db, catalog.product_id, store_cfg)
store = Store(paths.store_db)
print("Huella del estado inicial:", huella_inicial)
store.summary()

In [ ]:
# Ver la distribución de precios y stock generados.
inv = pd.read_sql("SELECT * FROM products", store.con)
inv["precio_eur"] = inv.price_cents / 100
print(inv.precio_eur.describe(percentiles=[0.1, 0.5, 0.9, 0.99]).round(2).to_string())
pd.cut(inv.stock, [-1, 0, 3, 40], labels=["agotado (0)", "bajo (1–3)", "normal (4–40)"]) \
    .value_counts(normalize=True).sort_index().map("{:.1%}".format).to_frame("proporción")

In [ ]:
# Comprobar la reproducibilidad: regenerar con la misma semilla da exactamente el mismo inventario.
store.close()
assert restore_store(paths.store_db, catalog.product_id, store_cfg) == huella_inicial
print("Restauración reproducible ✓")
store = Store(paths.store_db)

## 2. Operaciones de la tienda (lo que usarán las herramientas del agente)

| Operación | Valida | Devuelve |
|---|---|---|
| `get_price_stock(ids)` | que el producto exista | precio y stock **actuales** |
| `set_cart_qty(cart, id, qty)` | entero ≥ 0, ≤ 50, producto existente, qty ≤ stock | carrito actualizado (qty = 0 quita la línea) |
| `create_quote(cart, budget)` | carrito no vacío, stock y total ≤ presupuesto | `quote_id` con precios congelados |
| `confirm_order(quote_id)` | **re-verifica** stock, precio y presupuesto en una transacción | pedido, o conflictos sin cambiar nada |

Cada operación devuelve un `Result` con `ok`, `data` y `errors`. Los errores tienen un `code` (`insufficient_stock`, `over_budget`, `stock_changed`…) que el agente traduce a una respuesta en español.

Para las demostraciones se eligen tres productos del catálogo con stock conocido.

In [ ]:
# Elegir productos de ejemplo: uno con stock holgado, uno con stock bajo y uno agotado.
def elegir(cond):
    pid = inv[cond].product_id.iloc[0]
    return pid, catalog.set_index("product_id").title[pid][:70]

(P_OK, t1), (P_BAJO, t2), (P_AGOTADO, t3) = (
    elegir(inv.stock >= 10), elegir(inv.stock == 2), elegir(inv.stock == 0)
)
pd.DataFrame(store.get_price_stock([P_OK, P_BAJO, P_AGOTADO]).data["items"]).assign(
    titulo=[t1, t2, t3]
)

### 2.1 Validaciones del carrito

In [ ]:
# Probar cantidad inválida, más que el stock y producto agotado: se rechazan antes de escribir nada.
for pid, qty in [(P_OK, -2), (P_BAJO, 5), (P_AGOTADO, 1), ("NO_EXISTE", 1)]:
    r = store.set_cart_qty("demo", pid, qty)
    print(f"{pid:>12} qty={qty:>2} → ok={r.ok}  {r.errors}")

In [ ]:
# Cambiar la cantidad (escenario 3): fijar la cantidad reemplaza la anterior.
store.set_cart_qty("demo", P_OK, 2)
store.set_cart_qty("demo", P_OK, 3)
store.set_cart_qty("demo", P_BAJO, 1)
carrito = store.view_cart("demo")
print("Total:", carrito["total"])
pd.DataFrame(carrito["lines"])

### 2.2 Presupuesto

In [ ]:
# Probar un presupuesto menor al total: la cotización se rechaza e indica cuánto se excede.
presupuesto = carrito["total_cents"] - 500
r = store.create_quote("demo", budget_cents=presupuesto)
print(f"Presupuesto {fmt_eur(presupuesto)} → ok={r.ok}")
print(r.errors)

### 2.3 Escenario 6: cambio de stock entre la cotización y la confirmación

1. Se cotiza el carrito.
2. **Evento externo:** otro cliente compra y el stock de un producto baja por debajo de lo cotizado.
3. La confirmación re-verifica, detecta el conflicto y **no aplica nada**: tampoco descuenta los productos que sí tenían stock.
4. El agente propone ajustar (aquí: reducir la cantidad), re-cotiza y confirma.
5. El usuario repite "confirmo": se devuelve **el mismo pedido**, sin duplicarlo.

In [ ]:
cot = store.create_quote("demo", budget_cents=carrito["total_cents"] + 1000)
q1 = cot.data["quote_id"]
print("Cotización:", q1, cot.data["total"])

huella_antes = store.fingerprint()
store.set_stock(P_BAJO, 0)                      # simular un evento externo: se agotó
huella_evento = store.fingerprint()

r = store.confirm_order(q1)
print("Confirmación →", r.ok, r.errors)
assert store.fingerprint() == huella_evento     # nada cambió: sin cambios parciales
assert store.summary()["orders"] == 0
print("Sin cambios parciales ✓")

In [ ]:
# Quitar el producto agotado (lo propone el agente), aceptar, re-cotizar y confirmar.
store.set_cart_qty("demo", P_BAJO, 0)
q2 = store.create_quote("demo", budget_cents=carrito["total_cents"] + 1000).data["quote_id"]
r1 = store.confirm_order(q2)
r2 = store.confirm_order(q2)                    # repetir "confirmo", como haría el usuario
print("1ª confirmación:", r1.data)
print("2ª confirmación:", r2.data)
assert r1.data["order_id"] == r2.data["order_id"] and r2.data["duplicate"]
assert store.summary()["orders"] == 1
print("Sin pedidos duplicados ✓")

In [ ]:
# Verificar que el stock se descontó una sola vez.
pd.DataFrame(store.get_price_stock([P_OK]).data["items"])

## 3. Bitácora de operaciones

Cada escenario de evaluación guardará esta tabla junto con los audios, las transcripciones y el estado final, como pide el enunciado. Las filas `external_*` son eventos inyectados por la prueba, no acciones del agente.

In [ ]:
store.ops()[["seq", "op", "args", "ok"]]

## 4. Restaurar antes de seguir

Cada escenario parte del estado inicial: se restaura la tienda y se comprueba la huella.

In [ ]:
store.close()
assert restore_store(paths.store_db, catalog.product_id, store_cfg) == huella_inicial
store = Store(paths.store_db)
print(store.summary())
store.close()

### Resumen

- Tienda con 11 463 productos, precios y stock generados con semilla, y restauración verificada por huella.
- Validaciones de cantidad, stock y presupuesto en código, con errores estructurados para el agente.
- Confirmación atómica (*todo o nada*) e idempotente (*sin duplicados*), demostrada con el escenario 6.
- **Siguiente:** `03_recuperacion.ipynb` (BM25 como línea base frente a búsqueda densa, híbrida y con re-ranker, con nDCG@10).